# Text-to-SQL Transformer - Training on Kaggle

Trains the encoder-decoder Transformer from *Attention Is All You Need* on WikiSQL,
from scratch, using the code in this GitHub repository.

**Spec settings used here:** `d_model=256`, 4 heads, 3/3 layers, `d_ff=1024`,
dropout 0.1, post-norm, weight sharing, batch size **64**, **20 epochs**,
label smoothing 0.1, Adam (0.9, 0.98, 1e-9), warmup 4000.

**Flow:** clone repo -> fetch WikiSQL data -> prep data -> train -> verify -> artifacts.

**Before running (Kaggle notebook settings, right panel):**

1. **Settings -> Accelerator -> GPU** (T4 x2 or P100 — the code trains on one GPU).
2. **Settings -> Internet -> On** (needed for the `git clone` and the data download).
3. Run the cells in order.

## 0. Configuration

Set your GitHub repo URL here. Kaggle persists everything under `/kaggle/working`
as the notebook's Output, so no Drive mount is needed.

In [ ]:
import shutil
from pathlib import Path

REPO_URL = "https://github.com/MuhammadShaafImran/text-2-SQL-scratch.git"
REPO_DIR = "/kaggle/working/text-2-sql"

#Copied here at the end so the Output tab contains the checkpoints alone.
ARTIFACTS_DIR = "/kaggle/working/artifacts"

# For a PRIVATE repo, prefix the URL with a personal-access token:
# REPO_URL = "https://<username>:<ghp_xxx>@github.com/MuhammadShaafImran/text-2-SQL-scratch.git"

# WikiSQL source of truth.
WIKISQL_REPO = "https://github.com/salesforce/WikiSQL.git"
WIKISQL_DATA_TARBALL = (
    "https://github.com/salesforce/WikiSQL/raw/master/data.tar.bz2"
)

# train.py hyper-parameters (spec Task 3.5). Edit only if you know why.
EPOCHS = 20
BATCH_SIZE = 64

# Suffix for the checkpoint and training-log filenames. Leave it empty for the
# spec run, which writes best_model.pt / results/train_log.jsonl. Training past
# the 20-epoch budget is an ablation, not the headline: raise EPOCHS and set
# RUN_TAG="_60ep" so that run's artifacts cannot overwrite the spec run's.
RUN_TAG = ""

# Every later cell reads these, so there is one place that decides the names.
BEST_MODEL = f"best_model{RUN_TAG}.pt"
LAST_MODEL = f"last_model{RUN_TAG}.pt"
TRAIN_LOG = f"results/train_log{RUN_TAG}.jsonl"

if not shutil.which("git"):
    raise RuntimeError("git not found - this session has no git; use a Kaggle image with git")
print("config ready")

## 1. Verify the GPU

Training on CPU would take days. Make sure a GPU accelerator is selected.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Notebook settings (right panel) -> Accelerator -> GPU T4 x2 or P100."
    )

properties = torch.cuda.get_device_properties(0)
total_memory = getattr(properties, "total_memory", None)
if total_memory is None:  # older torch versions call it total_mem
    total_memory = properties.total_mem

print("GPU:", properties.name)
print("Memory:", round(total_memory / 1e9, 1), "GB")
if torch.cuda.device_count() > 1:
    print(f"{torch.cuda.device_count()} GPUs visible; training uses GPU 0")

## 2. Install dependencies

`records`, `babel`, `tqdm` and `tabulate` are needed by the official WikiSQL evaluator.
`torch`, `matplotlib` and `flask` are already in the Kaggle image.

In [ ]:
%pip install -q sentencepiece records babel tqdm tabulate

## 3. Clone the code and the WikiSQL evaluator

The model code comes from this repository. `salesforce/WikiSQL` is cloned only for
`evaluate.py` + `lib/` (the official evaluator required by section 3.1); the
dataset itself is downloaded as a tarball in the next cell.

The clone must contain `checks.py` and `tables.py` — commit and push those from
your local machine first, or cells 8 and 11 will fail.

In [ ]:
import os
import shutil
from pathlib import Path

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

# The clone lands inside /kaggle/working, which Kaggle saves automatically.
!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Official evaluator: only evaluate.py and lib/ are needed, and checking out the
# whole WikiSQL tree fails on filenames containing carriage returns.
if not Path("WikiSQL/evaluate.py").exists():
    !git clone --depth 1 {WIKISQL_REPO} WikiSQL
    !git -C WikiSQL checkout HEAD -- evaluate.py lib

required_code = ["train.py", "checks.py", "tables.py", "analysis.py",
                 "decode.py", "evaluate_model.py", "starter/check_starter.py"]
missing_code = [p for p in required_code if not Path(p).exists()]
if missing_code:
    raise FileNotFoundError(
        "The cloned repo is missing files - push your latest commits first: "
        + str(missing_code)
    )

print("repo:", os.getcwd())
print("official evaluator:", Path("WikiSQL/evaluate.py").exists())

## 4. Download and place the WikiSQL data

The tarball unpacks to `data/`, but this repository reads from `Wikidata/`,
so the extracted files are moved across. Skipped entirely if the data is already there.

In [ ]:
import shutil
import subprocess
from pathlib import Path

# The tarball unpacks to data/; this repo reads from Wikidata/.
if not Path('Wikidata/dev.db').exists():
    subprocess.run(
        ['curl', '-L', '-o', 'data.tar.bz2', WIKISQL_DATA_TARBALL], check=True
    )
    subprocess.run(['tar', 'xjf', 'data.tar.bz2'], check=True)
    extracted = Path('data')
    if extracted.exists():
        Path('Wikidata').mkdir(parents=True, exist_ok=True)
        # Move file by file rather than renaming the folder, so a partial
        # Wikidata/ is completed instead of the fresh extract being deleted.
        for item in extracted.iterdir():
            target = Path('Wikidata') / item.name
            if target.exists():
                target.unlink()
            shutil.move(str(item), str(target))
        shutil.rmtree(extracted)
    Path('data.tar.bz2').unlink(missing_ok=True)
else:
    print('WikiSQL data already in place - skipping the download')

required = [
    'Wikidata/train.jsonl', 'Wikidata/train.tables.jsonl', 'Wikidata/train.db',
    'Wikidata/dev.jsonl',   'Wikidata/dev.tables.jsonl',   'Wikidata/dev.db',
    'Wikidata/test.jsonl',  'Wikidata/test.tables.jsonl',  'Wikidata/test.db',
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f'Missing WikiSQL files: {missing}')

for p in required:
    print(f'{p:34s} {Path(p).stat().st_size / 1e6:8.1f} MB')


## 5. Build the pairs and the shared BPE vocabulary

`data/*.jsonl` and `data/sql_sp.model` are gitignored, so they are always rebuilt
here. One 8000-piece vocabulary is shared by source and target, as in the paper.

In [ ]:
# Run as modules so the repo root stays on sys.path and `starter.*` resolves.
!python -m starter.data_prep
!python -m starter.tokenizer

In [ ]:
# Sanity check: expect (B, S) / (B, T) token batches, then
# (B, S, 256) encoder input and (B, T-1, 256) decoder input.
!python -m starter.check_starter

## 6. Train

20 epochs, batch size 64, from random initialisation.

Each epoch prints train loss, dev loss, the current learning rate and grad norm,
and appends the same row to `results/train_log.jsonl` (used later for the loss
and learning-rate figures). `best_model.pt` keeps the epoch with the lowest dev
loss; `last_model.pt` is the most recent epoch.

**Runtime:** roughly 30-60 minutes on a Kaggle T4/P100.

In [ ]:
from pathlib import Path
import argparse

import train as train_module

# Pass an explicit Namespace instead of letting train.py read sys.argv: under
# Jupyter sys.argv is the kernel's own, not this cell's. main() then uses exactly
# the EPOCHS/BATCH_SIZE/RUN_TAG set above, so notebook and script cannot drift.
args = argparse.Namespace(epochs=EPOCHS, batch_size=BATCH_SIZE, run_tag=RUN_TAG)
print(f"epochs={args.epochs}  batch_size={args.batch_size}  run_tag={args.run_tag!r}")

train_module.main(args)


## 7. Verify the run

Confirms the checkpoint looks like a real training run before trusting it:
20 epochs logged, dev loss actually fell, no NaNs.

In [ ]:
import json
import math
from pathlib import Path

log_path = Path(TRAIN_LOG)
if not log_path.exists():
    raise FileNotFoundError(f"{TRAIN_LOG} missing - did training run?")

log = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
epochs = [row["epoch"] for row in log]

assert len(log) == EPOCHS, f"expected {EPOCHS} epochs, logged {len(log)}"
assert epochs == list(range(1, EPOCHS + 1)), f"non-contiguous epochs: {epochs}"
assert all(
    math.isfinite(row["train_loss"]) and math.isfinite(row["dev_loss"])
    for row in log
), "NaN/inf loss detected"
assert min(row["dev_loss"] for row in log) < log[0]["dev_loss"], "dev loss did not improve"

best = min(log, key=lambda row: row["dev_loss"])
print(f"epochs logged : {len(log)}")
print(f"best epoch    : {best['epoch']} (dev loss {best['dev_loss']:.4f})")
print(f"final epoch   : train {log[-1]['train_loss']:.4f} / dev {log[-1]['dev_loss']:.4f}")
print(f"total GPU time : {sum(row['seconds'] for row in log) / 60:.1f} min")
print(f"final LR      : {log[-1]['learning_rate']:.3e}")
print(f"GPU           : {log[-1].get('gpu', 'unknown')}")

## 8. Correctness checks (assignment section 3.2)

All six must pass. The gold round-trip is the one that catches evaluator bugs:
feeding the gold targets through your parser and the official evaluator must
score above 99% execution accuracy.

In [ ]:
import os

# The official evaluator opens files without an encoding; force UTF-8.
os.environ["PYTHONUTF8"] = "1"

# Run as a subprocess rather than %run: checks.py exits non-zero on failure,
# which would abort the cell and skip the artifact copy further down.
!python checks.py

## 9. Copy artifacts into the Kaggle Output folder

Kaggle automatically saves `/kaggle/working` as the notebook's Output (up to ~20 GB),
so the checkpoints survive the session. Re-run this cell any time; click
**Save Version** when training completes.

In [ ]:
import shutil
from pathlib import Path

destination = Path(ARTIFACTS_DIR)
destination.mkdir(parents=True, exist_ok=True)

artifacts = [
    BEST_MODEL,
    LAST_MODEL,
    TRAIN_LOG,
    "data/sql_sp.model",
    "data/sql_sp.vocab",
]

for artifact in artifacts:
    source = Path(artifact)
    if source.exists():
        shutil.copy2(source, destination / source.name)
        print(f"copied {artifact} -> {destination / source.name}")
    else:
        print(f"skipped {artifact} (not found)")

## 10. Evaluate (optional, runs in this session)

Writes one prediction line per dev row into `results/` and reports the official
WikiSQL logical-form and execution accuracy plus the component accuracies from
Task 5.2.

Decode all 8,421 dev rows - do **not** pass `--limit` when producing numbers for
the report. The test split should be run exactly once, at the very end:

```python
!python evaluate_model.py --split data/test_pairs.jsonl --strategy beam --beam-size 4
```

In [ ]:
os.environ["PYTHONUTF8"] = "1"

# dev, greedy
!python evaluate_model.py --split data/dev_pairs.jsonl --strategy greedy --checkpoint {BEST_MODEL}

# dev, beam search with beam size 4
!python evaluate_model.py --split data/dev_pairs.jsonl --strategy beam --beam-size 4 --checkpoint {BEST_MODEL}


## 11. Tables, figures and samples

Writes `results/tables.md` (Tables 1-4), the loss and learning-rate figures,
and `results/samples.md` (5 correct + 5 wrong dev examples).

In [ ]:
%run tables.py --log {TRAIN_LOG} --checkpoint {BEST_MODEL}

from pathlib import Path
print("\n--- results/tables.md ---")
print(Path("results/tables.md").read_text())


## 12. Analysis figures (optional)

Positional-encoding heatmap, the cross-attention map for one dev example,
and `results/analysis_report.md`.

In [ ]:
%run analysis.py --checkpoint {BEST_MODEL} --example 0

## 13. Package everything into the Output

Zips `results/` next to the checkpoints. In the notebook's **Output** tab you can
then download any of them (or the whole `text-2-sql/` folder) individually.

In [ ]:
import shutil
from pathlib import Path

destination = Path(ARTIFACTS_DIR)

# results/ is bulky; zip it so the Output tab has a single tidy file.
if Path("results").exists():
    shutil.make_archive("results", "zip", "results")
    shutil.move("results.zip", destination / "results.zip")
    print("archived results/ ->", destination / "results.zip")

for path in sorted(destination.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(destination)}  ({path.stat().st_size / 1e3:.1f} KB)")